# LeetCode 1536 — Minimum Swaps to Arrange a Binary Grid bằng Python

Link đề: https://leetcode.com/problems/minimum-swaps-to-arrange-a-binary-grid/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Matrix

Skeleton LeetCode:

```python
class Solution:
    def minSwaps(self, grid: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Biến bài toán ma trận thành bài toán **mảng một chiều**: mỗi hàng chỉ cần một con số — **số số 0 liên tiếp ở cuối hàng** (trailing zeros).
2. Viết baseline **BFS trên các hoán vị hàng** để có đáp án chuẩn với `n` nhỏ.
3. Nắm lời giải **tham lam**: với vị trí `i` từ trên xuống, chọn **hàng gần nhất** thoả yêu cầu rồi "nổi bọt" lên bằng các phép đổi kề nhau; trả `-1` nếu không có.
4. Chứng minh được tham lam tối ưu (lập luận "hàng lên đỉnh phải vượt qua mọi hàng phía trên nó" + quy nạp).
5. Biết kiểm tra nhanh tính khả thi bằng sắp xếp (gợi ý của đề) và tối ưu phần đếm bằng Fenwick tree + segment tree (mở rộng).

## 2. Đề bài & ví dụ

Cho lưới nhị phân `n × n`. Mỗi bước được chọn **hai hàng kề nhau** và **đổi chỗ** chúng. Lưới **hợp lệ** nếu mọi ô **phía trên đường chéo chính** đều bằng 0 (tức `grid[i][j] = 0` với mọi `j > i`). Trả về số bước ít nhất để lưới hợp lệ, hoặc `-1` nếu không thể.

**Ví dụ 1:** `grid = [[0,0,1],[1,1,0],[1,0,0]]` → `3`.

**Ví dụ 2:** `grid = [[0,1,1,0],[0,1,1,0],[0,1,1,0],[0,1,1,0]]` → `-1` (mọi hàng giống nhau, đổi chỗ không thay đổi gì).

**Ví dụ 3:** `grid = [[1,0,0],[1,1,0],[1,1,1]]` → `0`.

**Ràng buộc:** `n == grid.length == grid[i].length`, `1 ≤ n ≤ 200`, `grid[i][j]` là 0 hoặc 1.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [
    ([[0, 0, 1], [1, 1, 0], [1, 0, 0]], 3),
    ([[0, 1, 1, 0], [0, 1, 1, 0], [0, 1, 1, 0], [0, 1, 1, 0]], -1),
    ([[1, 0, 0], [1, 1, 0], [1, 1, 1]], 0),
]

def show(grid):
    n = len(grid)
    for i, row in enumerate(grid):
        cells = " ".join(("[" + str(v) + "]") if j > i else (" " + str(v) + " ") for j, v in enumerate(row))
        print(f"hàng {i}: {cells}")

for g, want in examples:
    show(g)
    print("-> mong đợi", want, "\n")

hàng 0:  0  [0] [1]
hàng 1:  1   1  [0]
hàng 2:  1   0   0 
-> mong đợi 3 

hàng 0:  0  [1] [1] [0]
hàng 1:  0   1  [1] [0]
hàng 2:  0   1   1  [0]
hàng 3:  0   1   1   0 
-> mong đợi -1 

hàng 0:  1  [0] [0]
hàng 1:  1   1  [0]
hàng 2:  1   1   1 
-> mong đợi 0 



Trong hình, các ô trong ngoặc vuông `[ ]` là ô **phía trên đường chéo** — chúng phải bằng 0.

## 3. Kiến thức nền

### 3.1 Đếm số 0 ở cuối hàng (trailing zeros)

Duyệt từ cuối hàng về đầu, đếm tới khi gặp số 1. Hàng `i` hợp lệ ⇔ các ô `j > i` đều 0 ⇔ số 0 cuối hàng `≥ n − 1 − i`.

In [2]:
def trailing_zeros(row):
    cnt = 0
    for v in reversed(row):
        if v == 1:
            break
        cnt += 1
    return cnt

for row in [[1, 0, 0], [0, 1, 0], [0, 0, 1], [0, 0, 0], [1, 1, 1]]:
    print(row, "->", trailing_zeros(row))

[1, 0, 0] -> 2
[0, 1, 0] -> 1
[0, 0, 1] -> 0
[0, 0, 0] -> 3
[1, 1, 1] -> 0


Một cách viết gọn khác: tìm vị trí số 1 **cuối cùng** (`maxRight` trong gợi ý của đề). Nếu không có số 1 thì coi là `-1`. Khi đó `trailing = n − 1 − maxRight`, và hàng đặt ở vị trí `i` hợp lệ ⇔ `maxRight ≤ i`.

In [3]:
def max_right(row):
    for j in range(len(row) - 1, -1, -1):
        if row[j] == 1:
            return j
    return -1

for row in [[1, 0, 0], [0, 1, 0], [0, 0, 1], [0, 0, 0]]:
    print(row, "maxRight =", max_right(row), "| trailing =", len(row) - 1 - max_right(row))

[1, 0, 0] maxRight = 0 | trailing = 2
[0, 1, 0] maxRight = 1 | trailing = 1
[0, 0, 1] maxRight = 2 | trailing = 0
[0, 0, 0] maxRight = -1 | trailing = 3


### 3.2 Đổi chỗ hai phần tử kề nhau và "nổi bọt"

Đưa phần tử ở vị trí `j` lên vị trí `i < j` bằng các phép đổi kề nhau cần đúng `j − i` bước, và **giữ nguyên thứ tự tương đối** của các phần tử `i..j−1` (chúng cùng dịch xuống 1). Trong Python: `a.insert(i, a.pop(j))` cho cùng kết quả.

In [4]:
a = ["A", "B", "C", "D"]
b = list(a)
for k in range(3, 0, -1):          # nổi bọt D từ vị trí 3 lên vị trí 0
    b[k - 1], b[k] = b[k], b[k - 1]
    print("đổi", k - 1, k, "->", b)
c = list(a)
c.insert(0, c.pop(3))
print("insert/pop:", c, "| giống nhau:", b == c)

đổi 2 3 -> ['A', 'B', 'D', 'C']
đổi 1 2 -> ['A', 'D', 'B', 'C']
đổi 0 1 -> ['D', 'A', 'B', 'C']
insert/pop: ['D', 'A', 'B', 'C'] | giống nhau: True


### 3.3 BFS trên không gian trạng thái

Mỗi trạng thái là một **thứ tự hàng** (tuple). Từ một trạng thái, có `n − 1` trạng thái kề (đổi hai hàng kề nhau). BFS cho số bước ít nhất tới trạng thái hợp lệ. Số trạng thái là `n!` — chỉ dùng được với `n ≤ 7`.

In [5]:
start = (0, 1, 2)
seen, q = {start}, deque([(start, 0)])
while q:
    st, d = q.popleft()
    for k in range(len(st) - 1):
        nx = list(st); nx[k], nx[k + 1] = nx[k + 1], nx[k]; nx = tuple(nx)
        if nx not in seen:
            seen.add(nx); q.append((nx, d + 1))
print("số trạng thái của 3 hàng:", len(seen))

số trạng thái của 3 hàng: 6


### 3.4 Biểu thức sinh với `next(..., default)`

`next((j for j in range(i, n) if cond(j)), -1)` trả về chỉ số **đầu tiên** thoả điều kiện, hoặc `-1` nếu không có — gọn cho bước "tìm hàng gần nhất".

In [6]:
z = [0, 1, 2]
print(next((j for j in range(0, 3) if z[j] >= 2), -1), next((j for j in range(0, 3) if z[j] >= 5), -1))

2 -1


## 4. Ý tưởng nền tảng / trực giác

Chuyển ví dụ 1 sang mảng số 0 cuối hàng: `[[0,0,1],[1,1,0],[1,0,0]]` → `z = [0, 1, 2]`. Yêu cầu theo vị trí (n = 3): hàng 0 cần `≥ 2`, hàng 1 cần `≥ 1`, hàng 2 cần `≥ 0` → `need = [2, 1, 0]` (giảm dần!).

Tính tay theo kiểu tham lam:

1. Vị trí 0 cần `≥ 2`: hàng gần nhất thoả là chỉ số 2 (`z = 2`). Nổi bọt lên: 2 bước → `z = [2, 0, 1]`.
2. Vị trí 1 cần `≥ 1`: trong `[0, 1]` (chỉ số 1, 2), gần nhất thoả là chỉ số 2. 1 bước → `z = [2, 1, 0]`.
3. Vị trí 2 cần `≥ 0`: luôn thoả.

Tổng `2 + 1 = 3`. Vì yêu cầu **giảm dần** khi đi xuống, một hàng đủ tốt cho vị trí trên thì cũng đủ tốt cho mọi vị trí dưới — nên "dùng hàng gần nhất" không bao giờ làm hỏng tương lai.

In [7]:
g = examples[0][0]
z = [trailing_zeros(r) for r in g]
n = len(g)
print("z    =", z)
print("need =", [n - 1 - i for i in range(n)])

z    = [0, 1, 2]
need = [2, 1, 0]


## 5. Cách 1 — Brute Force / baseline: BFS trên thứ tự hàng

Trạng thái = tuple thứ tự các hàng gốc. Đích = trạng thái mà hàng ở vị trí `i` có `z ≥ n − 1 − i` với mọi `i`. BFS từ `(0, 1, …, n−1)`.

In [8]:
def brute_bfs(grid):
    n = len(grid)
    z = [trailing_zeros(r) for r in grid]
    def ok(st):
        return all(z[st[i]] >= n - 1 - i for i in range(n))
    start = tuple(range(n))
    if ok(start):
        return 0
    seen, q = {start}, deque([(start, 0)])
    while q:
        st, d = q.popleft()
        for k in range(n - 1):
            nx = list(st)
            nx[k], nx[k + 1] = nx[k + 1], nx[k]
            nx = tuple(nx)
            if nx not in seen:
                if ok(nx):
                    return d + 1
                seen.add(nx)
                q.append((nx, d + 1))
    return -1

for g, want in examples:
    print(brute_bfs(g), "mong đợi", want)

3 mong đợi 3
-1 mong đợi -1
0 mong đợi 0


**Độ phức tạp:** `O(n! · n)` thời gian và bộ nhớ — `n = 8` đã là 40 320 trạng thái, `n = 200` thì vô vọng. Đúng là baseline để **kiểm chứng**.

**Nút thắt:** BFS thử mọi thứ tự hàng, trong khi cấu trúc "yêu cầu giảm dần" cho phép quyết định **từng vị trí một** mà không cần quay lui.

In [9]:
g6 = [[0] * 5 + [1]] + [[0] * k + [1] + [0] * (5 - k) for k in range(5)]
t0 = time.perf_counter()
print("n = 6:", brute_bfs(g6), f"({time.perf_counter() - t0:.3f}s)")

n = 6: 5 (0.000s)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — Nén mỗi hàng thành một số:** `z[r]` = số 0 cuối hàng `r`. Hàng `r` đặt ở vị trí `i` hợp lệ ⇔ `z[r] ≥ need[i] = n − 1 − i`.

**Insight 2 — Yêu cầu giảm dần:** `need[0] > need[1] > … > need[n−1] = 0`. Nên hàng đủ tốt cho vị trí `i` thì đủ tốt cho mọi vị trí `> i`.

**Quy tắc tham lam** (với `i = 0, 1, …, n−1`):
- **Nếu** không có hàng nào trong `z[i..n−1]` đạt `need[i]` **thì** trả `-1`.
- **Ngược lại** lấy `j` là chỉ số **nhỏ nhất** `≥ i` với `z[j] ≥ need[i]`; cộng `j − i` bước; nổi bọt hàng `j` lên vị trí `i` (các hàng `i..j−1` dịch xuống 1, giữ thứ tự).

**Invariant:** trước khi xử lý vị trí `i`, các vị trí `0..i−1` đã hợp lệ và không bao giờ bị động tới nữa; `z[i..n−1]` là các hàng còn lại theo đúng thứ tự tương đối ban đầu.

## 7. Các cách tiếp theo

### Cách 2 — Tham lam, đổi chỗ thật trên lưới

Làm đúng như mô tả nhưng thao tác **trên chính lưới**: kiểm tra hàng bằng cách đếm số 0 cuối mỗi lần, đổi chỗ từng cặp hàng kề nhau. Dễ hình dung nhưng chậm: mỗi lần kiểm tra hàng `O(n)`.

In [10]:
def greedy_grid(grid):
    g = [row[:] for row in grid]          # không sửa input
    n = len(g)
    steps = 0
    for i in range(n):
        need = n - 1 - i
        j = i
        while j < n and trailing_zeros(g[j]) < need:
            j += 1
        if j == n:
            return -1
        while j > i:                       # nổi bọt bằng các phép đổi kề nhau
            g[j - 1], g[j] = g[j], g[j - 1]
            j -= 1
            steps += 1
    return steps

for g, want in examples:
    print(greedy_grid(g), "mong đợi", want)

3 mong đợi 3
-1 mong đợi -1
0 mong đợi 0


**Độ phức tạp:** tìm hàng: tối đa `n` hàng × `O(n)` mỗi hàng ⇒ `O(n²)` mỗi vị trí ⇒ `O(n³)` tổng (`8·10⁶` với `n = 200` — vẫn qua, nhưng lãng phí). Space `O(n²)` cho bản sao.

### Cách 3 — Tham lam trên mảng `z`, `O(n²)`

Tính `z` một lần (`O(n²)` — bắt buộc vì phải đọc lưới), rồi tham lam trên mảng số nguyên:
- `j = next(...)`: hàng gần nhất thoả.
- `z.insert(i, z.pop(j))`: nổi bọt trong `O(n)`.

In [11]:
def greedy_z(grid):
    n = len(grid)
    z = [trailing_zeros(r) for r in grid]
    steps = 0
    for i in range(n):
        need = n - 1 - i
        j = next((j for j in range(i, n) if z[j] >= need), -1)
        if j == -1:
            return -1
        steps += j - i
        z.insert(i, z.pop(j))
    return steps

for g, want in examples:
    print(greedy_z(g), "mong đợi", want)

3 mong đợi 3
-1 mong đợi -1
0 mong đợi 0


In [12]:
class SolutionZ:
    def minSwaps(self, grid: List[List[int]]) -> int:
        n = len(grid)
        z = []
        for row in grid:
            c = 0
            for v in reversed(row):
                if v:
                    break
                c += 1
            z.append(c)
        ans = 0
        for i in range(n):
            need = n - 1 - i
            j = i
            while j < n and z[j] < need:
                j += 1
            if j == n:
                return -1
            ans += j - i
            z.insert(i, z.pop(j))
        return ans

print([SolutionZ().minSwaps(g) for g, _ in examples])

[3, -1, 0]


**Độ phức tạp:** Time `O(n²)` (đọc lưới `O(n²)` + `n` lần tìm/insert `O(n)`), Space `O(n)`.

### Cách 4 — Kiểm tra khả thi bằng sắp xếp (gợi ý của đề) + tham lam

Trước khi mô phỏng, có thể trả lời ngay `-1`: sắp xếp `maxRight` tăng dần; tồn tại cách xếp hợp lệ ⇔ `sorted_maxRight[i] ≤ i` với mọi `i` (đặt hàng "dễ" nhất lên trên… ngược lại: hàng có `maxRight` nhỏ nhất lên trên cùng). Đây là điều kiện cần và đủ; sau đó tham lam chắc chắn không gặp `-1`.

In [13]:
def feasible(grid):
    mr = sorted(max_right(r) for r in grid)
    return all(mr[i] <= i for i in range(len(mr)))

class SolutionSortCheck:
    def minSwaps(self, grid: List[List[int]]) -> int:
        if not feasible(grid):
            return -1
        n = len(grid)
        mr = [max_right(r) for r in grid]
        ans = 0
        for i in range(n):
            j = i
            while mr[j] > i:              # chắc chắn tìm thấy vì đã khả thi
                j += 1
            ans += j - i
            mr.insert(i, mr.pop(j))
        return ans

for g, want in examples:
    print("khả thi:", feasible(g), "| đáp án:", SolutionSortCheck().minSwaps(g), "mong đợi", want)

khả thi: True | đáp án: 3 mong đợi 3
khả thi: False | đáp án: -1 mong đợi -1
khả thi: True | đáp án: 0 mong đợi 0


**Độ phức tạp:** như cách 3, thêm `O(n log n)` cho sắp xếp. Lợi ích: tách bạch "có làm được không" và "tốn bao nhiêu".

### Cách 5 — Không dịch mảng: segment tree tìm hàng + Fenwick đếm, `O(n log n)` sau khi đọc lưới

Quan sát: khi xử lý vị trí `i`, chi phí `j − i` chính là **số hàng chưa dùng đứng trước hàng được chọn trong thứ tự gốc**. Vậy không cần dịch mảng:
- **Segment tree** lưu `max z` của các hàng chưa dùng; tìm chỉ số gốc **nhỏ nhất** có `z ≥ need` bằng cách đi xuống cây (`O(log n)`), rồi đánh dấu đã dùng (gán `-1`).
- **Fenwick tree** đếm số hàng chưa dùng có chỉ số gốc `< j` (`O(log n)`).

Với `n ≤ 200` thì không cần, nhưng đây là kỹ thuật chuẩn khi `n` lớn (ví dụ bài "số lần đổi kề nhau tối thiểu" tổng quát).

In [14]:
class SolutionTree:
    def minSwaps(self, grid: List[List[int]]) -> int:
        n = len(grid)
        z = [trailing_zeros(r) for r in grid]
        size = 1
        while size < n:
            size *= 2
        seg = [-1] * (2 * size)
        for i, v in enumerate(z):
            seg[size + i] = v
        for p in range(size - 1, 0, -1):
            seg[p] = max(seg[2 * p], seg[2 * p + 1])
        bit = [0] * (n + 1)                       # Fenwick: 1 = hàng chưa dùng
        def bit_add(i, d):
            i += 1
            while i <= n:
                bit[i] += d
                i += i & -i
        def bit_sum(i):                           # tổng trên [0, i)
            s = 0
            while i > 0:
                s += bit[i]
                i -= i & -i
            return s
        for i in range(n):
            bit_add(i, 1)
        ans = 0
        for i in range(n):
            need = n - 1 - i
            if seg[1] < need:
                return -1
            p = 1
            while p < size:                       # đi xuống nhánh trái nếu đủ
                p = 2 * p if seg[2 * p] >= need else 2 * p + 1
            j = p - size
            ans += bit_sum(j)                     # số hàng chưa dùng đứng trước j
            bit_add(j, -1)
            seg[p] = -1
            p //= 2
            while p:
                seg[p] = max(seg[2 * p], seg[2 * p + 1])
                p //= 2
        return ans

print([SolutionTree().minSwaps(g) for g, _ in examples])

[3, -1, 0]


**Độ phức tạp:** `O(n²)` để đọc lưới (không tránh được) + `O(n log n)` cho phần tham lam; Space `O(n)`.

## 8. TRACE từng bước

Lưới 4 × 4 có `z = [0, 3, 1, 2]` (`need = [3, 2, 1, 0]`):

| i | need | z trước | j (gần nhất thoả) | bước thêm | z sau | tổng |
|---|------|---------|-------------------|-----------|-------|------|
| 0 | 3 | [0,3,1,2] | 1 | 1 | [3,0,1,2] | 1 |
| 1 | 2 | [3,0,1,2] | 3 | 2 | [3,2,0,1] | 3 |
| 2 | 1 | [3,2,0,1] | 3 | 1 | [3,2,1,0] | 4 |
| 3 | 0 | [3,2,1,0] | 3 | 0 | [3,2,1,0] | 4 |

Hàm `trace_greedy` in bảng này và lưới sau cùng.

In [15]:
def make_grid_from_z(z):
    n = len(z)
    rows = []
    for t in z:
        row = [0] * n
        if t < n:
            row[n - 1 - t] = 1            # số 1 ngay trước khối 0 cuối hàng
        rows.append(row)
    return rows

def trace_greedy(grid):
    n = len(grid)
    z = [trailing_zeros(r) for r in grid]
    order = list(range(n))
    total = 0
    print(f"{'i':>2} | need | {'z trước':14} | j  | +bước | {'z sau':14} | tổng")
    for i in range(n):
        need = n - 1 - i
        before = list(z)
        j = next((j for j in range(i, n) if z[j] >= need), -1)
        if j == -1:
            print(f"{i:>2} | {need:>4} | {str(before):14} | không có hàng thoả -> -1")
            return -1
        total += j - i
        z.insert(i, z.pop(j))
        order.insert(i, order.pop(j))
        print(f"{i:>2} | {need:>4} | {str(before):14} | {j:<2} | {j - i:>5} | {str(z):14} | {total}")
    print("Lưới sau khi xếp:")
    show([grid[r] for r in order])
    return total

g4 = make_grid_from_z([0, 3, 1, 2])
show(g4)
print()
print("Kết quả:", trace_greedy(g4))

hàng 0:  0  [0] [0] [1]
hàng 1:  1   0  [0] [0]
hàng 2:  0   0   1  [0]
hàng 3:  0   1   0   0 

 i | need | z trước        | j  | +bước | z sau          | tổng
 0 |    3 | [0, 3, 1, 2]   | 1  |     1 | [3, 0, 1, 2]   | 1
 1 |    2 | [3, 0, 1, 2]   | 3  |     2 | [3, 2, 0, 1]   | 3
 2 |    1 | [3, 2, 0, 1]   | 3  |     1 | [3, 2, 1, 0]   | 4
 3 |    0 | [3, 2, 1, 0]   | 3  |     0 | [3, 2, 1, 0]   | 4
Lưới sau khi xếp:
hàng 0:  1  [0] [0] [0]
hàng 1:  0   1  [0] [0]
hàng 2:  0   0   1  [0]
hàng 3:  0   0   0   1 
Kết quả: 4


In [16]:
print("Trace ví dụ 2 (không khả thi):")
trace_greedy(examples[1][0])

Trace ví dụ 2 (không khả thi):
 i | need | z trước        | j  | +bước | z sau          | tổng
 0 |    3 | [1, 1, 1, 1]   | không có hàng thoả -> -1


-1

## 9. Code nộp LeetCode

Bản nộp chính: cách 3, đúng chữ ký `minSwaps(self, grid: List[List[int]]) -> int`.

In [17]:
class Solution:
    def minSwaps(self, grid: List[List[int]]) -> int:
        n = len(grid)
        zeros = []
        for row in grid:                       # số 0 liên tiếp ở cuối mỗi hàng
            c = 0
            for v in reversed(row):
                if v == 1:
                    break
                c += 1
            zeros.append(c)
        ans = 0
        for i in range(n):
            need = n - 1 - i                   # vị trí i cần ít nhất n-1-i số 0 cuối
            j = i
            while j < n and zeros[j] < need:   # hàng gần nhất thoả
                j += 1
            if j == n:
                return -1
            ans += j - i                       # nổi bọt hàng j lên vị trí i
            zeros.insert(i, zeros.pop(j))
        return ans

for g, want in examples:
    got = Solution().minSwaps(g)
    print(got, "OK" if got == want else "SAI")

3 OK
-1 OK
0 OK


## 10. Vì sao thuật toán đúng

Gọi `T(A)` là số bước tối thiểu cho dãy hàng `A` (vị trí `0..m−1`, yêu cầu `need_i` giảm dần). Gọi `j` là chỉ số nhỏ nhất có `z ≥ need_0`.

**Chặn dưới.** Xét một lời giải tối ưu bất kỳ, gọi `R` là hàng kết thúc ở **vị trí 0**, chỉ số gốc `r`. Vì các hàng có chỉ số `< j` không đạt `need_0`, ta có `r ≥ j`. Hàng `R` cuối cùng nằm trên tất cả `r` hàng vốn ở phía trên nó, mà mỗi phép đổi kề chỉ vượt qua đúng một hàng ⇒ có **ít nhất `r` phép đổi liên quan tới `R`**. Các phép đổi còn lại (không chạm `R`), nếu xoá `R` khỏi dãy, vẫn là các phép đổi kề nhau và đưa dãy `A − R` về trạng thái hợp lệ cho vị trí `1..m−1` ⇒ có ít nhất `T(A − R)` phép. Vậy `OPT ≥ r + T(A − R)`.

**Tham lam không tệ hơn.** Tham lam tốn `j + T(A − j)`. Từ dãy `A − j`, ta có thể dùng `r − 1 − j` phép đổi để đưa `R` về đúng chỗ mà `j` chiếm trong `A − R`; khi đó dãy giống `A − R` nhưng `j` được thay bằng `R` — mà `R` có `z ≥ need_0 ≥` mọi yêu cầu còn lại, nên cùng chuỗi phép đổi tối ưu của `A − R` vẫn cho kết quả hợp lệ. Do đó `T(A − j) ≤ (r − 1 − j) + T(A − R)` (khi `r > j`), suy ra `j + T(A − j) ≤ r − 1 + T(A − R) < OPT` — mâu thuẫn trừ khi `r = j`. Vậy chọn hàng gần nhất là tối ưu; quy nạp cho các vị trí sau.

**Trường hợp `-1`.** Nếu ở bước `i` không còn hàng nào đạt `need_i`, thì các hàng còn lại (vốn phải lấp các vị trí `i..n−1`) đều có `z < need_i`; nhưng các vị trí trên đã dùng những hàng đạt yêu cầu lớn hơn, nên **không có** cách xếp nào để vị trí `i` hợp lệ: đếm số hàng có `z ≥ need_i` cần ít nhất `i + 1` hàng (cho các vị trí `0..i`) mà chỉ có `i` ⇒ bất khả thi.

## 11. Phân tích Time / Space

Gọi `n` = kích thước lưới (đầu vào có `n²` ô).

| Cách | Time | Space | Ghi chú |
|------|------|-------|---------|
| 1. BFS hoán vị | `O(n! · n)` | `O(n! · n)` | chỉ `n ≤ 7` |
| 2. Tham lam trên lưới | `O(n³)` | `O(n²)` | đếm lại số 0 mỗi lần |
| 3. Tham lam trên `z` | `O(n²)` | `O(n)` | **bản nộp chính** |
| 4. Kiểm tra sắp xếp + tham lam | `O(n²)` | `O(n)` | tách bạch khả thi / chi phí |
| 5. Segment tree + Fenwick | `O(n²)` đọc + `O(n log n)` | `O(n)` | kỹ thuật cho `n` lớn |

Với mọi cách tham lam, `O(n²)` để đọc lưới là cận dưới (phải xem các ô cuối hàng; worst case hàng toàn 0).

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, lưới toàn 0, lưới toàn 1, lưới đã hợp lệ, lưới ngược hoàn toàn), đối chiếu với BFS trên lưới ngẫu nhiên `n ≤ 6`, so các cách tham lam với nhau trên `n ≤ 60`, và đo thời gian `n = 200`.

In [18]:
def random_grid(n, rng, mode="z"):
    if mode == "cells":
        return [[rng.randint(0, 1) for _ in range(n)] for _ in range(n)]
    rows = []
    for _ in range(n):
        t = rng.randint(0, n)                     # số 0 cuối hàng mong muốn
        row = [rng.randint(0, 1) for _ in range(n)]
        for c in range(n - t, n):
            row[c] = 0
        if t < n:
            row[n - 1 - t] = 1
        rows.append(row)
    return rows

METHODS = {
    "greedy_grid": greedy_grid,
    "greedy_z": greedy_z,
    "SolutionZ": lambda g: SolutionZ().minSwaps(g),
    "SolutionSortCheck": lambda g: SolutionSortCheck().minSwaps(g),
    "SolutionTree": lambda g: SolutionTree().minSwaps(g),
    "Solution": lambda g: Solution().minSwaps(g),
}
edge = [([[0]], 0), ([[1]], 0), ([[0, 0], [0, 0]], 0), ([[1, 1], [1, 1]], -1),
        ([[1, 1], [1, 0]], 1), ([[1, 0], [1, 1]], 0),
        (make_grid_from_z([0, 1, 2]), 3), (make_grid_from_z([0, 1, 2, 3]), 6),
        (make_grid_from_z([3, 0, 2, 3]), 2), (make_grid_from_z([1, 1, 1]), -1)]
for g, want in examples + edge:
    assert brute_bfs(g) == want, g
    for name, f in METHODS.items():
        got = f(g)
        assert got == want, (name, g, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [19]:
rng = random.Random(1536)
cnt_neg = 0
for _ in range(400):
    n = rng.randint(1, 6)
    g = random_grid(n, rng, rng.choice(["z", "z", "cells"]))
    ref = brute_bfs(g)
    cnt_neg += ref == -1
    for name, f in METHODS.items():
        assert f(g) == ref, (name, g)
print(f"400 lưới ngẫu nhiên n ≤ 6 (seed 1536): khớp BFS ({cnt_neg} trường hợp -1)")

for _ in range(200):
    n = rng.randint(7, 60)
    g = random_grid(n, rng)
    ref = greedy_z(g)
    for name, f in METHODS.items():
        assert f(g) == ref, (name, n)
print("200 lưới n = 7..60: mọi cách tham lam khớp nhau")

400 lưới ngẫu nhiên n ≤ 6 (seed 1536): khớp BFS (132 trường hợp -1)


200 lưới n = 7..60: mọi cách tham lam khớp nhau


In [20]:
rng2 = random.Random(200)
z_perm = list(range(200))
rng2.shuffle(z_perm)
big = make_grid_from_z(z_perm)                    # luôn khả thi
worst = make_grid_from_z(list(range(200)))       # đảo ngược hoàn toàn: n(n-1)/2 bước
for name in ["greedy_grid", "greedy_z", "SolutionTree", "Solution"]:
    t0 = time.perf_counter()
    res = [METHODS[name](big), METHODS[name](worst)]
    print(f"{name:13s}: {res} ({time.perf_counter() - t0:.3f}s)")
    assert res[1] == 200 * 199 // 2

greedy_grid  : [9111, 19900] (0.056s)
greedy_z     : [9111, 19900] (0.002s)
SolutionTree : [9111, 19900] (0.002s)
Solution     : [9111, 19900] (0.003s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm **tổng** số 0 của hàng thay vì số 0 **ở cuối**

Hàng `[0, 1, 0]` có 2 số 0 nhưng chỉ 1 số 0 ở cuối — đặt ở vị trí 0 sẽ có ô `(0,1) = 1` phía trên đường chéo.

In [21]:
def wrong_total_zeros(grid):
    n = len(grid)
    z = [row.count(0) for row in grid]            # SAI: phải là số 0 ở cuối
    ans = 0
    for i in range(n):
        j = next((j for j in range(i, n) if z[j] >= n - 1 - i), -1)
        if j == -1:
            return -1
        ans += j - i
        z.insert(i, z.pop(j))
    return ans

tests = examples + [([[0, 1, 0], [1, 1, 0], [1, 1, 1]], -1)]
for g, want in tests:
    print(g, "| sai:", wrong_total_zeros(g), "| đúng:", Solution().minSwaps(g))

[[0, 0, 1], [1, 1, 0], [1, 0, 0]] | sai: 0 | đúng: 3
[[0, 1, 1, 0], [0, 1, 1, 0], [0, 1, 1, 0], [0, 1, 1, 0]] | sai: -1 | đúng: -1
[[1, 0, 0], [1, 1, 0], [1, 1, 1]] | sai: 0 | đúng: 0
[[0, 1, 0], [1, 1, 0], [1, 1, 1]] | sai: 0 | đúng: -1


Sửa: đếm từ cuối hàng tới số 1 đầu tiên (`trailing_zeros`).

### Lỗi 2 — Chọn hàng có **nhiều số 0 nhất** thay vì hàng **gần nhất** thoả

Hàng "dư" số 0 không cần thiết ở vị trí hiện tại; kéo nó từ xa lên tốn thêm bước.

In [22]:
def wrong_pick_max(grid):
    n = len(grid)
    z = [trailing_zeros(r) for r in grid]
    ans = 0
    for i in range(n):
        j = max(range(i, n), key=lambda t: z[t])  # SAI: chọn hàng nhiều 0 nhất
        if z[j] < n - 1 - i:
            return -1
        ans += j - i
        z.insert(i, z.pop(j))
    return ans

g = make_grid_from_z([3, 0, 2, 3])
print("z = [3, 0, 2, 3] | sai:", wrong_pick_max(g), "| đúng:", Solution().minSwaps(g))

z = [3, 0, 2, 3] | sai: 3 | đúng: 2


Sửa: chọn chỉ số **nhỏ nhất** `j ≥ i` thoả `z[j] ≥ need`.

### Lỗi 3 — Đổi chỗ trực tiếp hàng `i` và hàng `j` (không kề nhau) nhưng vẫn cộng `j − i`

Đổi trực tiếp đưa hàng `i` xuống tận vị trí `j`, làm sai thứ tự các hàng còn lại — trạng thái này **không** đạt được bằng `j − i` phép đổi kề nhau.

In [23]:
def wrong_direct_swap(grid):
    n = len(grid)
    z = [trailing_zeros(r) for r in grid]
    ans = 0
    for i in range(n):
        j = next((j for j in range(i, n) if z[j] >= n - 1 - i), -1)
        if j == -1:
            return -1
        ans += j - i
        z[i], z[j] = z[j], z[i]                    # SAI: phải nổi bọt (insert/pop)
    return ans

for zz in ([0, 1, 2], [0, 1, 2, 3]):
    g = make_grid_from_z(zz)
    print("z =", zz, "| sai:", wrong_direct_swap(g), "| đúng:", Solution().minSwaps(g), "| BFS:", brute_bfs(g))

z = [0, 1, 2] | sai: 2 | đúng: 3 | BFS: 3
z = [0, 1, 2, 3] | sai: 4 | đúng: 6 | BFS: 6


Sửa: `z.insert(i, z.pop(j))` (hoặc đổi kề nhau từng bước).

### Lỗi 4 — Không xử lý trường hợp không tìm thấy hàng (quên `-1`)

Nếu bỏ qua vị trí không tìm được hàng phù hợp (hoặc lấy `j = i` mặc định), ta trả về một con số vô nghĩa thay vì `-1`.

In [24]:
def wrong_no_minus_one(grid):
    n = len(grid)
    z = [trailing_zeros(r) for r in grid]
    ans = 0
    for i in range(n):
        j = next((j for j in range(i, n) if z[j] >= n - 1 - i), i)   # SAI: mặc định i
        ans += j - i
        z.insert(i, z.pop(j))
    return ans

for g, want in examples:
    print("sai:", wrong_no_minus_one(g), "| đúng:", want)

sai: 3 | đúng: 3
sai: 0 | đúng: -1
sai: 0 | đúng: 0


Sửa: không tìm thấy ⇒ `return -1` ngay.

### Lỗi 5 — Lệch một trong yêu cầu: dùng `need = n − i`

Hàng `i` chỉ cần các ô `j > i` bằng 0, tức `n − 1 − i` ô. Dùng `n − i` đòi hỏi cả ô trên đường chéo cũng bằng 0 ⇒ trả `-1` cho lưới hợp lệ.

In [25]:
def wrong_need(grid):
    n = len(grid)
    z = [trailing_zeros(r) for r in grid]
    ans = 0
    for i in range(n):
        j = next((j for j in range(i, n) if z[j] >= n - i), -1)    # SAI: n - i
        if j == -1:
            return -1
        ans += j - i
        z.insert(i, z.pop(j))
    return ans

for g, want in examples:
    print("sai:", wrong_need(g), "| đúng:", want)

sai: -1 | đúng: 3
sai: -1 | đúng: -1
sai: -1 | đúng: 0


Sửa: `need = n − 1 − i`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Ma trận + thao tác trên **cả hàng** ⇒ nén mỗi hàng thành **một đặc trưng** (ở đây: số 0 cuối hàng / vị trí số 1 cuối).
- "Đổi chỗ **kề nhau** tối thiểu" ⇒ nghĩ tới **nổi bọt** và **số nghịch thế**: đưa phần tử từ `j` lên `i` tốn đúng `j − i`.
- Yêu cầu **đơn điệu** theo vị trí (vị trí trên khắt khe hơn) ⇒ tham lam lấp vị trí khắt khe trước bằng ứng viên **gần nhất**.
- Kiểm tra khả thi kiểu "sắp xếp rồi so `a[i] ≤ i`" là mẫu Hall's condition đơn giản.
- Bài cùng họ: *Minimum Adjacent Swaps to Reach the Kth Smallest Number* (1850), *Minimum Number of Moves to Make Palindrome* (2193), *Minimum Swaps to Group All 1's Together* (1151), *Count of Smaller Numbers After Self* (315 — Fenwick).

## 15. Biến thể của bài

1. **Các ô dưới đường chéo** phải bằng 0: đếm số 0 ở **đầu** hàng, yêu cầu tăng dần ⇒ tham lam từ dưới lên (hoặc đảo lưới).
2. **Đổi cột kề nhau** thay vì hàng: chuyển vị lưới rồi giải như cũ (đường chéo chính bất biến qua chuyển vị, nhưng "trên" thành "dưới").
3. **Được đổi hai hàng bất kỳ** (không cần kề): chi phí mỗi lần là 1 ⇒ bài toán thành số chu trình của hoán vị — khác hẳn.
4. **Chỉ hỏi có làm được không**: sắp xếp `maxRight`, kiểm tra `a[i] ≤ i`.
5. **Ma trận chữ nhật / yêu cầu tuỳ ý** `need_i` không đơn điệu: tham lam gần nhất có thể sai; cần phân tích lại (ví dụ thành bài gán có chi phí nghịch thế).

In [26]:
def variant_below(grid):
    """Biến thể 1: các ô DƯỚI đường chéo phải bằng 0 -> lật ngược thứ tự hàng và cột rồi dùng lời giải gốc."""
    flipped = [row[::-1] for row in grid[::-1]]
    return Solution().minSwaps(flipped)

def brute_below(grid):
    n = len(grid)
    lead = []
    for row in grid:
        c = 0
        for v in row:
            if v: break
            c += 1
        lead.append(c)
    start = tuple(range(n))
    ok = lambda st: all(lead[st[i]] >= i for i in range(n))
    if ok(start): return 0
    seen, q = {start}, deque([(start, 0)])
    while q:
        st, d = q.popleft()
        for k in range(n - 1):
            nx = list(st); nx[k], nx[k + 1] = nx[k + 1], nx[k]; nx = tuple(nx)
            if nx not in seen:
                if ok(nx): return d + 1
                seen.add(nx); q.append((nx, d + 1))
    return -1

rngv = random.Random(15)
for _ in range(200):
    n = rngv.randint(1, 5)
    g = [row[::-1] for row in random_grid(n, rngv)]
    assert variant_below(g) == brute_below(g)
print("Biến thể 'dưới đường chéo': khớp BFS; ví dụ", variant_below([[0, 0, 1], [0, 1, 1], [1, 1, 1]]))

Biến thể 'dưới đường chéo': khớp BFS; ví dụ 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. BFS hoán vị | thử mọi thứ tự hàng | `O(n!·n)` | `O(n!·n)` | kiểm chứng `n ≤ 7` |
| 2. Tham lam trên lưới | nổi bọt thật, đếm 0 lại mỗi lần | `O(n³)` | `O(n²)` | minh hoạ trực quan |
| 3. Tham lam trên `z` | nén hàng thành số 0 cuối | `O(n²)` | `O(n)` | **bản nộp chính** |
| 4. Sắp xếp kiểm tra + tham lam | Hall's condition rồi mô phỏng | `O(n²)` | `O(n)` | cần trả lời nhanh `-1` |
| 5. Segment tree + Fenwick | đếm hàng chưa dùng đứng trước | `O(n²) + O(n log n)` | `O(n)` | khi phần tham lam là nút cổ chai |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minSwaps([[1,1,0],[1,0,0],[1,1,1]])` trả về bao nhiêu?

**Bài 2 (tính tay):** Với `z = [1, 0, 3, 2]` (n = 4), chạy tay tham lam: ghi `j` và số bước ở từng vị trí, tổng là bao nhiêu?

**Bài 3 (điền TODO):** Hoàn thiện `ex3_greedy(z)` nhận thẳng mảng số 0 cuối hàng và trả về số bước (hoặc `-1`).

**Bài 4 (tự viết lại):** Viết `ex4_feasible(grid)` chỉ trả `True/False` (có xếp được không) trong `O(n²)` mà **không** mô phỏng đổi chỗ.

**Bài 5 (mini challenge):** Viết `ex5_swaps(grid)` trả về **danh sách các phép đổi** `(r, r+1)` (theo chỉ số hàng hiện tại tại thời điểm đổi) có độ dài tối thiểu; áp dụng lên lưới phải ra lưới hợp lệ. Trả `None` nếu không thể.

In [27]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_greedy(z: List[int]) -> int:
    z = list(z)
    n = len(z)
    ans = 0
    for i in range(n):
        # TODO: tìm j gần nhất >= i với z[j] >= n - 1 - i; không có -> return -1
        # TODO: ans += j - i; nổi bọt z[j] lên vị trí i
        pass
    return ans

In [28]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_feasible(grid: List[List[int]]) -> bool:
    # TODO
    raise NotImplementedError

def ex5_swaps(grid: List[List[int]]):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tính `z`: `[1, 2, 0]`. (2) `need = [2, 1, 0]`: vị trí 0 lấy chỉ số 1 (1 bước) → `[2, 1, 0]`. (3) Các vị trí sau đã thoả ⇒ tổng 1.

**Bài 2.** (1) `need = [3, 2, 1, 0]`. (2) Vị trí 0: `j = 2` (2 bước) → `[3, 1, 0, 2]`; vị trí 1: `j = 3` (2 bước) → `[3, 2, 1, 0]`. (3) Tổng 4.

**Bài 3.** (1) Dùng vòng `while j < n and z[j] < need`. (2) `if j == n: return -1`. (3) `z.insert(i, z.pop(j))`.

**Bài 4.** (1) Tính `maxRight` mỗi hàng. (2) Sắp xếp tăng dần. (3) Khả thi ⇔ `mr[i] ≤ i` với mọi `i` (chứng minh: xếp hàng theo thứ tự đó là một cách hợp lệ; ngược lại nếu `mr[i] > i` thì chỉ có ≤ `i` hàng dùng được cho `i + 1` vị trí đầu).

**Bài 5.** (1) Mô phỏng như cách 2 trên **thứ tự hàng**. (2) Khi nổi bọt từ `j` về `i`, ghi lại `(j−1, j), (j−2, j−1), …, (i, i+1)`. (3) Kiểm tra độ dài = `minSwaps` và lưới sau khi áp dụng hợp lệ.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
a1 = Solution().minSwaps([[1, 1, 0], [1, 0, 0], [1, 1, 1]])
print("Bài 1:", a1)
assert a1 == 1

# Bài 2
def ex3_greedy_sol(z):
    z = list(z)
    n, ans = len(z), 0
    for i in range(n):
        need = n - 1 - i
        j = i
        while j < n and z[j] < need:
            j += 1
        if j == n:
            return -1
        ans += j - i
        z.insert(i, z.pop(j))
    return ans

a2 = ex3_greedy_sol([1, 0, 3, 2])
print("Bài 2:", a2)
assert a2 == 4 == brute_bfs(make_grid_from_z([1, 0, 3, 2]))

Bài 1: 1
Bài 2: 4


In [30]:
# Bài 3
rng3 = random.Random(3)
for _ in range(300):
    n = rng3.randint(1, 6)
    z = [rng3.randint(0, n) for _ in range(n)]
    assert ex3_greedy_sol(z) == brute_bfs(make_grid_from_z(z))
print("Bài 3: OK")

Bài 3: OK


In [31]:
# Bài 4
def ex4_feasible_sol(grid):
    mr = sorted(max_right(r) for r in grid)
    return all(mr[i] <= i for i in range(len(mr)))

rng4 = random.Random(4)
for _ in range(400):
    n = rng4.randint(1, 6)
    g = random_grid(n, rng4, rng4.choice(["z", "cells"]))
    assert ex4_feasible_sol(g) == (brute_bfs(g) != -1)
print("Bài 4: OK")

Bài 4: OK


In [32]:
# Bài 5
def ex5_swaps_sol(grid):
    n = len(grid)
    z = [trailing_zeros(r) for r in grid]
    ops = []
    for i in range(n):
        j = next((j for j in range(i, n) if z[j] >= n - 1 - i), -1)
        if j == -1:
            return None
        for k in range(j, i, -1):
            ops.append((k - 1, k))
            z[k - 1], z[k] = z[k], z[k - 1]
    return ops

def is_valid(grid):
    return all(grid[i][j] == 0 for i in range(len(grid)) for j in range(i + 1, len(grid)))

rng5 = random.Random(5)
for _ in range(300):
    n = rng5.randint(1, 8)
    g = random_grid(n, rng5)
    ops = ex5_swaps_sol(g)
    best = Solution().minSwaps(g)
    if ops is None:
        assert best == -1
        continue
    h = [row[:] for row in g]
    for a, b in ops:
        assert b == a + 1
        h[a], h[b] = h[b], h[a]
    assert is_valid(h) and len(ops) == best
print("Bài 5: OK, ví dụ 1:", ex5_swaps_sol(examples[0][0]))

Bài 5: OK, ví dụ 1: [(1, 2), (0, 1), (1, 2)]


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def minSwaps(self, grid):
        n = len(grid)
        z = []
        for row in grid:                      # số 0 ở CUỐI hàng
            c = 0
            for v in reversed(row):
                if v: break
                c += 1
            z.append(c)
        ans = 0
        for i in range(n):
            j = i
            while j < n and z[j] < n - 1 - i: # hàng GẦN NHẤT thoả
                j += 1
            if j == n: return -1
            ans += j - i
            z.insert(i, z.pop(j))             # nổi bọt, giữ thứ tự các hàng khác
        return ans
```

**Template "đổi kề nhau tối thiểu với yêu cầu đơn điệu":** nén mỗi phần tử thành đặc trưng → lấp vị trí khắt khe nhất trước bằng ứng viên gần nhất → chi phí = khoảng cách → `insert/pop` để giữ thứ tự.

**Checklist trước khi submit:**
- [ ] Đếm số 0 **ở cuối** hàng, không phải tổng số 0.
- [ ] `need = n − 1 − i` (không phải `n − i`).
- [ ] Chọn hàng **gần nhất** thoả, không phải hàng nhiều 0 nhất.
- [ ] Nổi bọt bằng `insert/pop`, không đổi trực tiếp `z[i], z[j]`.
- [ ] Không tìm thấy hàng ⇒ `-1` ngay.